# Sprint 7 - Data Preparation Pipeline

This notebook demonstrates the complete data preparation pipeline for traditional machine learning models.

Pipeline:

Raw Data
→ Validation
→ RUL Generation
→ Feature Engineering
→ Feature Selection
→ Train/Validation Split
→ Feature Scaling

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent

sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

A:\AI_Engineer\ML Projects\Predictive-Maintenance-RUL


In [8]:
import pandas as pd

from src.data.loader import DataLoader
from src.data.validator import DataValidator
from src.preprocessing.rul_generator import RULGenerator
from src.preprocessing.feature_engineer import FeatureEngineer

from src.preprocessing.feature_selector import FeatureSelector
from src.preprocessing.data_splitter import DataSplitter
from src.preprocessing.feature_scaler import FeatureScaler
from src.preprocessing.preparation_pipeline import DataPreparationPipeline
from src.utils.constant import SENSOR_COLUMNS

from src.config.config import (
    TRAIN_DATA_PATH,
    TEST_DATA_PATH,
    RUL_DATA_PATH
)

1- LOAD DATA

In [3]:
loader = DataLoader(
    train_path=TRAIN_DATA_PATH,
    test_path=TEST_DATA_PATH,
    rul_path=RUL_DATA_PATH,
)

train_df = loader.load_train()
test_df = loader.load_test()
rul_df = loader.load_rul()

2026-07-27 14:59:12 | INFO | loader.py | Line:18 | Reading train_FD004.txt
2026-07-27 14:59:14 | INFO | loader.py | Line:21 | train_FD004.txt Loaded Successfully
2026-07-27 14:59:14 | INFO | loader.py | Line:18 | Reading test_FD004.txt
2026-07-27 14:59:15 | INFO | loader.py | Line:21 | test_FD004.txt Loaded Successfully
2026-07-27 14:59:15 | INFO | loader.py | Line:18 | Reading RUL_FD004.txt
2026-07-27 14:59:15 | INFO | loader.py | Line:21 | RUL_FD004.txt Loaded Successfully


2- VALIDATE DATA

In [4]:
validator = DataValidator(
    train_df,
    test_df,
    rul_df
)

validator.validate_all()

2026-07-27 15:00:07 | INFO | validator.py | Line:40 | Validating training dataset...
2026-07-27 15:00:07 | INFO | validator.py | Line:50 | Validating testing dataset...
2026-07-27 15:00:07 | INFO | validator.py | Line:60 | Validating RUL dataset...


{'train': {'valid': True, 'errors': [], 'warnings': []},
 'test': {'valid': True, 'errors': [], 'warnings': []},
 'rul': {'valid': True, 'errors': [], 'warnings': ['Duplicate rows found.']}}

3- Generate RUL

In [6]:
generator = RULGenerator(train_df=train_df)

train_df = generator.generate(cap=125)

train_df.head()

2026-07-27 15:02:17 | INFO | rul_generator.py | Line:82 | Generating Remaining Useful Life (RUL)...
2026-07-27 15:02:17 | INFO | rul_generator.py | Line:92 | Applying RUL cap = 125
2026-07-27 15:02:17 | INFO | rul_generator.py | Line:96 | RUL generated successfully.


,unit_number,time_in_cycles,operational_setting_1,operational_setting_2,operational_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,RUL
0,1,1,42.0049,0.8400,100.0,445.00,549.68,1343.43,1112.93,3.91,...,2387.99,8074.83,9.3335,0.02,330,2212,100.00,10.62,6.3670,125
1,1,2,20.0020,0.7002,100.0,491.19,606.07,1477.61,1237.50,9.35,...,2387.73,8046.13,9.1913,0.02,361,2324,100.00,24.37,14.6552,125
2,1,3,42.0038,0.8409,100.0,445.00,548.95,1343.12,1117.05,3.91,...,2387.97,8066.62,9.4007,0.02,329,2212,100.00,10.48,6.4213,125
3,1,4,42.0000,0.8400,100.0,445.00,548.70,1341.24,1118.03,3.91,...,2388.02,8076.05,9.3369,0.02,328,2212,100.00,10.54,6.4176,125
4,1,5,25.0063,0.6207,60.0,462.54,536.10,1255.23,1033.59,7.05,...,2028.08,7865.80,10.8366,0.02,305,1915,84.93,14.03,8.6754,125


4- Feature Engineering

In [11]:
engineer = FeatureEngineer(sensor_columns=SENSOR_COLUMNS,lags=[1,2,3],)

feature_df = engineer.transform(train_df)
feature_df.head()

2026-07-27 15:07:17 | INFO | feature_engineer.py | Line:50 | Starting Feature Engineering...
2026-07-27 15:07:17 | INFO | feature_engineer.py | Line:119 | Generating Rolling Mean features...
2026-07-27 15:07:18 | INFO | feature_engineer.py | Line:148 | Generating Rolling Std features...
2026-07-27 15:07:19 | INFO | feature_engineer.py | Line:179 | Generating Lag Features...
A:\AI_Engineer\ML Projects\Predictive-Maintenance-RUL\src\preprocessing\feature_engineer.py:189: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[feature_name] = (
2026-07-27 15:07:19 | INFO | feature_engineer.py | Line:205 | Generating Rate of Change features...
A:\AI_Engineer\ML Projects\Predictive-Maintenance-RUL\src\preprocessing\feature_engineer.py:213: PerformanceWarning: DataFrame

,unit_number,time_in_cycles,operational_setting_1,operational_setting_2,operational_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_12_diff,sensor_13_diff,sensor_14_diff,sensor_15_diff,sensor_16_diff,sensor_17_diff,sensor_18_diff,sensor_19_diff,sensor_20_diff,sensor_21_diff
0,1,1,42.0049,0.8400,100.0,445.00,549.68,1343.43,1112.93,3.91,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,2,20.0020,0.7002,100.0,491.19,606.07,1477.61,1237.50,9.35,...,182.81,-0.26,-28.70,-0.1422,0.0,31.0,112.0,0.00,13.75,8.2882
2,1,3,42.0038,0.8409,100.0,445.00,548.95,1343.12,1117.05,3.91,...,-182.97,0.24,20.49,0.2094,0.0,-32.0,-112.0,0.00,-13.89,-8.2339
3,1,4,42.0000,0.8400,100.0,445.00,548.70,1341.24,1118.03,3.91,...,0.18,0.05,9.43,-0.0638,0.0,-1.0,0.0,0.00,0.06,-0.0037
4,1,5,25.0063,0.6207,60.0,462.54,536.10,1255.23,1033.59,7.05,...,34.31,-359.94,-210.25,1.4997,0.0,-23.0,-297.0,-15.07,3.49,2.2578


In [12]:
print(f"Rows      : {feature_df.shape[0]}")
print(f"Columns   : {feature_df.shape[1]}")

Rows      : 61249
Columns   : 153


5- Create Pipeline

In [13]:
pipeline = DataPreparationPipeline(

    splitter=DataSplitter(
        test_size=0.2,
    ),

    selector=FeatureSelector(

        target_column="RUL",

        drop_columns=[
            "unit_number",
        ],

    ),

    scaler=FeatureScaler(),

)

In [14]:
X_train, X_val, y_train, y_val = pipeline.prepare(
    feature_df
)

2026-07-27 15:09:17 | INFO | preparation_pipeline.py | Line:37 | Starting Data Preparation Pipeline...
2026-07-27 15:09:17 | INFO | data_splitter.py | Line:28 | Starting engine-based train/validation split...
2026-07-27 15:09:17 | INFO | data_splitter.py | Line:51 | Train Engines: 199 | Validation Engines: 50
2026-07-27 15:09:17 | INFO | data_splitter.py | Line:56 | Data splitting completed successfully.
2026-07-27 15:09:17 | INFO | feature_selector.py | Line:29 | Starting Feature Selection...
2026-07-27 15:09:17 | INFO | feature_selector.py | Line:40 | Feature Selection completed successfully.
2026-07-27 15:09:17 | INFO | feature_selector.py | Line:29 | Starting Feature Selection...
2026-07-27 15:09:17 | INFO | feature_selector.py | Line:40 | Feature Selection completed successfully.
2026-07-27 15:09:17 | INFO | feature_scaler.py | Line:33 | Fitting Feature Scaler...
2026-07-27 15:09:17 | INFO | feature_scaler.py | Line:37 | Feature Scaler fitted successfully.
2026-07-27 15:09:17 | IN

In [16]:
print("Training Features :", X_train.shape)
print("Validation Features :", X_val.shape,"\n")

print("Training Target :", y_train.shape)
print("Validation Target :", y_val.shape)

Training Features : (49072, 151)
Validation Features : (12177, 151) 

Training Target : (49072,)
Validation Target : (12177,)


In [19]:
X_train.describe().T[
    ["mean","std"]
].head(15).round(1)

,mean,std
time_in_cycles,-0.0,1.0
operational_setting_1,-0.0,1.0
operational_setting_2,0.0,1.0
operational_setting_3,0.0,1.0
sensor_1,0.0,1.0
sensor_2,-0.0,1.0
sensor_3,-0.0,1.0
sensor_4,0.0,1.0
sensor_5,0.0,1.0
sensor_6,0.0,1.0


In [20]:
display(X_train.head())

display(y_train.head())

,time_in_cycles,operational_setting_1,operational_setting_2,operational_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,sensor_6,...,sensor_12_diff,sensor_13_diff,sensor_14_diff,sensor_15_diff,sensor_16_diff,sensor_17_diff,sensor_18_diff,sensor_19_diff,sensor_20_diff,sensor_21_diff
0,-1.475980,1.218608,0.864911,0.419453,-1.054356,-0.795346,-0.699862,-0.744354,-1.137979,-1.081887,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,-1.464977,-0.271012,0.414701,0.419453,0.694359,0.715648,0.563946,0.299616,0.365060,0.372465,...,0.931295,-0.002303,-0.243161,-0.133863,-0.000062,0.786467,0.543757,-0.000855,0.976271,0.980598
2,-1.453974,1.218534,0.867809,0.419453,-1.054356,-0.814907,-0.702781,-0.709826,-1.137979,-1.083726,...,-0.930807,0.000461,0.169939,0.198596,-0.000062,-0.812224,-0.544361,-0.000855,-0.984687,-0.972654
3,-1.442971,1.218276,0.864911,0.419453,-1.054356,-0.821606,-0.720489,-0.701613,-1.137979,-1.081887,...,0.001567,-0.000589,0.077057,-0.059731,-0.000062,-0.025567,-0.000302,-0.000855,0.005015,0.000325
4,-1.431968,0.067784,0.158681,-2.384059,-0.390306,-1.159228,-1.530595,-1.409270,-0.270416,-0.475141,...,0.175315,-1.990352,-1.767825,1.418650,-0.000062,-0.583840,-1.443030,-1.989595,0.248361,0.267680


0    125
1    125
2    125
3    125
4    125
Name: RUL, dtype: int64

## Engineering Decisions

### Data Leakage Prevention

- Target generated before splitting.
- Feature Engineering performed independently for each engine.
- Train/Validation split performed by engine IDs.
- Scaler fitted only on the training data.

---

### Feature Selection

Removed:

- unit_number

Target:

- RUL

---

### Scaling

StandardScaler

Fit:

Training only

Transform:

Validation only

---

### Missing Values

NaN values generated by

- Rolling Mean
- Rolling Std
- Lag Features

These will be handled in the next sprint.

---

### Output

The pipeline now returns

X_train
X_validation
y_train
y_validation

ready for model training.